# Survival prediction on synthetic data

The simplest way to use the kalecancer Pipeline. Each patient has a small clinical table row and a bag of patch features (what a whole-slide image encoder would produce). They are fused and trained end to end with a Cox head. Runs on a CPU in under a minute and downloads nothing.

In [ ]:
import tempfile
from functools import partial
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler

from kalecancer.evaluate import HarrellC
from kalecancer.loaddata import MultimodalDataset, PatchFeatures, TimeToEvent, train_test_split
from kalecancer.model import ABMIL, Concat, CoxHead, IntermediateFusion
from kalecancer.pipeline import Pipeline

## Synthetic cohort

A clinical table, one h5 file of patch features per patient in a temporary folder, and censored survival times.

In [ ]:
# creating the synthetic data

N_PATIENTS = 300
PATCH_DIM = 32


def make_cohort(folder: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Write one h5 file of patch features per patient into ``folder``; return the clinical table and outcomes."""
    rng = np.random.default_rng(0)
    ids = pd.Index([f"patient_{k:03d}" for k in range(N_PATIENTS)], name="patient_id")
    clinical = pd.DataFrame({"age": rng.normal(60, 10, N_PATIENTS), "stage": rng.integers(1, 5, N_PATIENTS)}, index=ids)

    aggressive_fraction = rng.uniform(0, 0.5, N_PATIENTS)
    for patient, fraction in zip(ids, aggressive_fraction, strict=True):
        n_patches = int(rng.integers(20, 200))
        features = rng.normal(size=(n_patches, PATCH_DIM)).astype(np.float32)
        features[: int(fraction * n_patches), 0] += 3.0  # aggressive patches stand out in one feature
        with h5py.File(folder / f"{patient}.h5", "w") as f:
            f["features"] = features
            f["coords"] = rng.integers(0, 50_000, size=(n_patches, 2))

    risk = 0.09 * (clinical["age"] - 60) + 1.2 * (clinical["stage"] - 2.5) + 12.0 * aggressive_fraction
    event_time = rng.exponential(2000 * np.exp(-risk))
    censoring_time = rng.uniform(200, 3000, N_PATIENTS)
    outcome = pd.DataFrame(
        {"time": np.minimum(event_time, censoring_time), "event": event_time <= censoring_time}, index=ids
    )
    return clinical, outcome


workdir = tempfile.TemporaryDirectory()
folder = Path(workdir.name)
clinical, outcome = make_cohort(folder)

In [ ]:
clinical.index

Index(['patient_000', 'patient_001', 'patient_002', 'patient_003',
       'patient_004', 'patient_005', 'patient_006', 'patient_007',
       'patient_008', 'patient_009',
       ...
       'patient_290', 'patient_291', 'patient_292', 'patient_293',
       'patient_294', 'patient_295', 'patient_296', 'patient_297',
       'patient_298', 'patient_299'],
      dtype='object', name='patient_id', length=300)

## 1. Data

Modalities and a target, all indexed by patient id, and which modalities every patient must have.

In [ ]:
data = MultimodalDataset(

    # modalities provided as dict, with the key chosen by the user
    modalities={
        # a pandas DataFrame, with the index set as the patient_i
        "clinical": clinical,

        # from_glob = files from a given patter
        "wsi": PatchFeatures.from_glob(

            # all.h5 files in the provided folder is data (i.e. X matrix)
            f"{folder}/*.h5",

            # the patient_id comes from the file names. this regex describes
            # how to extract the patient_id from the file names (everything in
            # the brackets is used what is selected from the name as an id).
            # in this example, the patient ID is defined as "patient_xxx".
            id_pattern=r"(patient_\d{3})\.h5$"
        ),
    },

    # the prediction task
    target=TimeToEvent(time=outcome["time"], event=outcome["event"]),

    # what modalities must be present for each patient_id. In this example,
    # only patients that have both clinical and wsi datat will be kept.
    required_modalities=["clinical", "wsi"],
)

train, test = train_test_split(data, test_size=0.25, stratify=True, random_state=0)

data.summary()

,count
available: clinical,300
available: wsi,300
available: target,300
included,300
included with clinical,300
included with wsi,300
events,269


## 2. Model

Stages for each modality, a fusion method, and a prediction head.

In [ ]:
# Intermediate fusion: encode the modalities seperately, then fuse, 
# then prediction head.

model = IntermediateFusion(

    # as encoding in intermediate fusion is modality-specific, we set a dictionary
    # with the keys matching the ones in the MultimodalDataset object in the last cell
    encoding={

        "clinical": [torch.nn.Linear(2, 8)],

        # a list of encoders = run the encoders sequentially. So in this example
        # the output of the ABMIL must equal input of Linear layer.
        "wsi": [
            ABMIL(in_dim=PATCH_DIM, hidden_dim=16, attention_dim=8, dropout=0.0),
            torch.nn.Linear(16, 8),
        ],
    },

    # fusion method = concatenation in this example. So the output of the fusion
    # will be output dim of both encoders from above which is 8 (for clinical) +
    # 8 (for wsi) = 16 dim
    fusion=Concat(),

    # The in_dim of the prediction head must equal to the output of the fusion 
    # method. In this example, the output of concat is 16 dim.
    head=CoxHead(in_dim=16, ties="efron"),
)

## 3. Pipeline

Preprocessing and training settings. Transforms are 'trained' on the training patient only, and then used to transform the test data. For a validation split with early stopping, see `examples/hancock/survival_intermediate.py`.

In [ ]:
pipeline = Pipeline(
    model=model,
    transforms={"clinical": StandardScaler()}, # keys must match MultimodalDataset object keys
    optimizer=partial(torch.optim.AdamW, lr=1e-3),
    batch_size=32,
    drop_last=True,
    max_epochs=40,
    validation=None,
    early_stopping=None,
    accelerator="cpu",
    precision="32-true",
    random_state=0,
)

## 4. Train and evaluate

Fit on training data, and predict on test data.

In [ ]:
# fit on training data
pipeline.fit(train)

,model,IntermediateF...s=False) ) )
,transforms,{'clinical': StandardScaler()}
,optimizer,"functools.par...W'>, lr=0.001)"
,batch_size,32
,drop_last,True
,max_epochs,40
,validation,None
,early_stopping,None
,accelerator,'cpu'
,precision,'32-true'
,random_state,0


In [ ]:
# training epoch history
pipeline.history_ 

,train_loss,optimizer_steps,skipped_batches
epoch,,,
0,2.664669,7,0
1,2.654173,14,0
2,2.634469,21,0
3,2.620321,28,0
4,2.601041,35,0
5,2.599116,42,0
6,2.592169,49,0
7,2.563750,56,0
8,2.550561,63,0


In [ ]:
# report information on the run
# best_epoch and best_score are None, as current implementation only fetches
# this when early stopping and validation are provides in the Pipeline object.
# Adding support for these is a WIP.
pipeline.fit_report_

{'n_train': 225,
 'n_val': 0,
 'events': 202,
 'epochs_run': 40,
 'best_epoch': None,
 'best_score': None,
 'optimizer_steps': 280,
 'skipped_batches': 0}

In [ ]:
# evaluate on the test set, using chosen metric(s)
pipeline.evaluate(test, metrics={"harrell_c": HarrellC()})

harrell_c    0.791497
Name: score, dtype: float64

In [ ]:
# predict the log hazard for each patient in the test set
pipeline.predict(test).head()

,log_hazard
id,
patient_001,3.357584
patient_006,-0.994580
patient_011,-2.813317
patient_013,-0.559498
patient_019,-1.708847


In [ ]:
# cleaning up the synthetic data
workdir.cleanup()